In [ ]:
# Import required libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path
sys.path.append('..')

from src.transforms.xlet_nsst import XLETNSST
from src.analysis.feature_analysis import FeatureExtractor, FeatureEvaluator
from src.visualization.visualize import FeatureVisualizer

print("✓ Libraries imported successfully")

## 1. Load Test Image

In [ ]:
# Load image
image_path = '../data/00004.png'  # Change to your image
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f"Image shape: {image.shape}")
print(f"Data type: {image.dtype}")
print(f"Value range: [{image.min()}, {image.max()}]")

# Display
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title('Original Image')
plt.axis('off')
plt.show()

## 2. Apply XLET-NSST Transformation

In [ ]:
# Initialize transformer
transformer = XLETNSST(
    levels=3,          # Number of decomposition levels
    directions=8,      # Number of directional subbands
    shear_levels=2,
    filter_type='maxflat'
)

print("Applying XLET-NSST transformation...")
coeffs = transformer.transform(image)

# Count channels
num_channels = len([k for k in coeffs.keys() if isinstance(coeffs[k], np.ndarray)])
print(f"\n✓ Generated {num_channels} feature channels")

# Show channel names
print("\nAvailable channels:")
for idx, key in enumerate(coeffs.keys(), 1):
    if isinstance(coeffs[key], np.ndarray):
        print(f"  {idx}. {key}: shape {coeffs[key].shape}")

## 3. Visualize All Subbands

In [ ]:
visualizer = FeatureVisualizer(figsize=(16, 12))

# Visualize all subbands
fig = visualizer.visualize_all_subbands(coeffs, cmap='viridis')
plt.show()

## 4. Visualize Scale Decomposition

In [ ]:
# Show decomposition across scales and directions
fig = visualizer.visualize_scale_decomposition(coeffs)
plt.show()

## 5. Analyze Feature Statistics

In [ ]:
# Extract and analyze features
extractor = FeatureExtractor()
analysis = extractor.analyze_all_channels(coeffs)

print("Feature Analysis Complete!\n")
print("Sample statistics for 'lowpass' channel:")
if 'lowpass' in analysis:
    for metric, value in analysis['lowpass'].items():
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Visualize statistics
fig = visualizer.plot_feature_statistics(
    analysis, 
    metrics=['entropy', 'energy', 'std', 'dynamic_range']
)
plt.show()

## 6. Rank Features for Segmentation

In [ ]:
# Rank features
evaluator = FeatureEvaluator()
rankings = evaluator.rank_features_for_segmentation(coeffs)

print("Top 10 Feature Channels for Semantic Segmentation:\n")
for idx, (channel, score) in enumerate(rankings[:10], 1):
    print(f"{idx:2d}. {channel:30s} - Score: {score:.4f}")

In [ ]:
# Visualize rankings
fig = visualizer.visualize_ranking_results(
    rankings[:15],
    title="Top 15 Feature Channels for Segmentation"
)
plt.show()

## 7. Get Best Channels by Specific Metrics

In [ ]:
# Best by entropy (information content)
best_entropy = extractor.get_best_channels_by_metric(analysis, 'entropy', top_k=5)
print("Top 5 by Entropy (Information Content):")
for channel, value in best_entropy:
    print(f"  {channel}: {value:.4f}")

print()

# Best by energy
best_energy = extractor.get_best_channels_by_metric(analysis, 'energy', top_k=5)
print("Top 5 by Energy:")
for channel, value in best_energy:
    print(f"  {channel}: {value:.4f}")

print()

# Best by standard deviation (variance)
best_std = extractor.get_best_channels_by_metric(analysis, 'std', top_k=5)
print("Top 5 by Standard Deviation:")
for channel, value in best_std:
    print(f"  {channel}: {value:.4f}")

## 8. Select Diverse Channels

In [ ]:
# Select diverse channels with low correlation
diverse_channels = extractor.select_diverse_channels(
    coeffs, 
    num_channels=10,
    correlation_threshold=0.8
)

print("Selected Diverse Channels (low correlation):")
for idx, channel in enumerate(diverse_channels, 1):
    print(f"  {idx}. {channel}")

## 9. Visualize Channel Correlations

In [ ]:
# Compute correlation matrix
corr_matrix, channel_names = extractor.compute_channel_correlation(coeffs)

print(f"Correlation matrix shape: {corr_matrix.shape}")
print(f"Number of channels: {len(channel_names)}")

# Visualize (limited to first 30 channels for clarity)
fig = visualizer.plot_correlation_matrix(
    corr_matrix[:30, :30],
    channel_names[:30]
)
plt.show()

## 10. Compare Original with Best Channels

In [ ]:
# Show original vs top 6 channels
top_6_channels = [name for name, _ in rankings[:6]]

fig = visualizer.visualize_comparison(
    image,
    coeffs,
    top_6_channels
)
plt.show()

## 11. Create Feature Vector for Segmentation

In [ ]:
# Create a feature vector using selected channels
selected_for_segmentation = [name for name, _ in rankings[:10]]

feature_vector = extractor.create_feature_vector(
    coeffs,
    selected_channels=selected_for_segmentation,
    resize_shape=(256, 256)  # Resize all to same size
)

print(f"Feature vector shape: {feature_vector.shape}")
print(f"  Height: {feature_vector.shape[0]}")
print(f"  Width: {feature_vector.shape[1]}")
print(f"  Channels: {feature_vector.shape[2]}")
print(f"\nThis feature vector can be used as input to your segmentation model!")

## 12. Experiment with Different Parameters

In [ ]:
# Compare different parameter settings
configs = [
    {'levels': 2, 'directions': 4},
    {'levels': 3, 'directions': 8},
    {'levels': 4, 'directions': 8},
]

print("Comparing different configurations:\n")

for config in configs:
    transformer_test = XLETNSST(**config)
    coeffs_test = transformer_test.transform(image)
    num_ch = len([k for k in coeffs_test.keys() if isinstance(coeffs_test[k], np.ndarray)])
    
    print(f"Levels={config['levels']}, Directions={config['directions']}: {num_ch} channels")

## 13. Summary and Recommendations

In [ ]:
print("="*70)
print("XLET-NSST FEATURE ANALYSIS SUMMARY")
print("="*70)
print()
print(f"Total Feature Channels Generated: {num_channels}")
print()
print("TOP RECOMMENDED CHANNELS FOR SEMANTIC SEGMENTATION:")
print()
for idx, (channel, score) in enumerate(rankings[:5], 1):
    print(f"{idx}. {channel}")
    print(f"   Score: {score:.4f}")
    if channel in analysis:
        print(f"   Entropy: {analysis[channel]['entropy']:.4f}")
        print(f"   Energy: {analysis[channel]['energy']:.4f}")
    print()

print("="*70)
print("NEXT STEPS:")
print("  1. Use these channels as input features for your segmentation model")
print("  2. Test on your specific dataset to validate performance")
print("  3. Fine-tune selection based on your segmentation results")
print("  4. Consider combining with other features (e.g., RGB, texture)")
print("="*70)